# Portfolio optimization with feedback strategies based on artificial neural networks

**Authors**: Y. Kopeliovich, Michael Pokojovy

**Published**: 2024-10-01

**arXiv**: [https://arxiv.org/abs/2411.09899](https://arxiv.org/abs/2411.09899)

With the recent advancements in machine learning (ML), artificial neural networks (ANN) are starting to play an increasingly important role in quantitative finance. Dynamic portfolio optimization is among many problems that have significantly benefited from a wider adoption of deep learning (DL). While most existing research has primarily focused on how DL can alleviate the curse of dimensionality when solving the Hamilton-Jacobi-Bellman (HJB) equation, some very recent developments propose to forego derivation and solution of HJB in favor of empirical utility maximization over dynamic allocation strategies expressed through ANN. In addition to being simple and transparent, this approach is universally applicable, as it is essentially agnostic about market dynamics. To showcase the method, we apply it to optimal portfolio allocation between a cash account and the S&P 500 index modeled using geometric Brownian motion or the Heston model. In both cases, the results are demonstrated to be on par with those under the theoretical optimal weights assuming isoelastic utility and real-time rebalancing. A set of R codes for a broad class of stochastic volatility models are provided as a supplement.

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1: Configuration

In this phase, we define the universe of assets, parameters for the strategy, and the hypothesis that we aim to test. This setup will guide the subsequent phases of our analysis.

In [ ]:
import numpy as np
import pandas as pd

# Configuration
universe = ['SPY']  # S&P 500 ETF as a proxy for the index
params = {
    'lookback_period': 252,  # 1 year of trading days
    'rebalance_frequency': 'monthly'
}
hypothesis = "ANN-based dynamic allocation strategies can optimize portfolio performance compared to static allocations."

## Phase 2: Data Download and Feature Computation

We will download historical price data for the assets in our universe using yfinance. Then, we compute relevant features and normalize them for further analysis.

In [ ]:
import yfinance as yf

# Download data
data = yf.download(universe, start='2010-01-01', end='2023-01-01')['Adj Close']

# Feature computation: daily returns
returns = data.pct_change().dropna()

# Normalization (z-score)
normalized_returns = (returns - returns.mean()) / returns.std()

## Phase 3: Signal Generation and Portfolio Construction

In this phase, we generate trading signals based on the computed features and construct the portfolio by determining position sizes.

In [ ]:
# Signal generation: simple moving average crossover as a placeholder
short_window = 40
long_window = 100

signals = pd.DataFrame(index=returns.index)
signals['signal'] = 0.0
signals['short_mavg'] = returns.rolling(window=short_window, min_periods=1, center=False).mean()
signals['long_mavg'] = returns.rolling(window=long_window, min_periods=1, center=False).mean()
signals['signal'][short_window:] = np.where(signals['short_mavg'][short_window:] > signals['long_mavg'][short_window:], 1.0, 0.0)

# Position sizing
signals['positions'] = signals['signal'].diff()

## Phase 4: Vectorized Backtest

We perform a backtest of the strategy by shifting signals to avoid look-ahead bias and calculate the strategy's performance.

In [ ]:
# Backtest
initial_capital = 100000.0
positions = pd.DataFrame(index=signals.index).fillna(0.0)
portfolio = pd.DataFrame(index=signals.index)

positions['SPY'] = signals['signal'] * initial_capital
portfolio['positions'] = (positions.multiply(returns, axis=0)).sum(axis=1)
portfolio['cash'] = initial_capital - (signals['positions'] * positions['SPY']).cumsum()
portfolio['total'] = portfolio['positions'] + portfolio['cash']

## Phase 5: Performance Metrics and Visualization

We evaluate the strategy using various performance metrics such as Sharpe ratio, Sortino ratio, Calmar ratio, and maximum drawdown. We also plot the equity curve.

In [ ]:
import matplotlib.pyplot as plt

# Performance metrics
def calculate_performance_metrics(portfolio):
    returns = portfolio['total'].pct_change().dropna()
    sharpe_ratio = np.sqrt(252) * returns.mean() / returns.std()
    downside_returns = returns[returns < 0]
    sortino_ratio = np.sqrt(252) * returns.mean() / downside_returns.std()
    max_drawdown = (portfolio['total'] / portfolio['total'].cummax() - 1).min()
    calmar_ratio = returns.mean() / abs(max_drawdown)
    return sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown

sharpe, sortino, calmar, max_dd = calculate_performance_metrics(portfolio)

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(portfolio['total'], label='Portfolio value')
plt.title('Equity Curve')
plt.xlabel('Date')
plt.ylabel('Portfolio Value')
plt.legend()
plt.show()

print(f"Sharpe Ratio: {sharpe:.2f}")
print(f"Sortino Ratio: {sortino:.2f}")
print(f"Calmar Ratio: {calmar:.2f}")
print(f"Max Drawdown: {max_dd:.2%}")

## Phase 6: Monitoring Stub

This phase includes a stub for monitoring daily P&L and positions, which can be expanded for real-time monitoring.

In [ ]:
# Monitoring stub
def monitor_portfolio(current_data, signals):
    # Placeholder for real-time P&L and position monitoring
    current_returns = current_data.pct_change().iloc[-1]
    current_positions = signals['signal'].iloc[-1]
    daily_pnl = current_positions * current_returns
    return daily_pnl, current_positions

# Example usage
# daily_pnl, current_positions = monitor_portfolio(latest_data, signals)
# print(f"Daily P&L: {daily_pnl}, Current Positions: {current_positions}")